# Text and layout

This notebook builds a small layout with `Card`, `Column`, `Row`, `Text`, and `Divider`. It has no interactive controls, so each cell finishes on its own.

Run the setup cell first. It detects the kernel and sets `KERNEL`, then defines `new_surface`, which sends `createSurface` with the v1.0 basic catalog. Every notebook in this directory uses the same setup cell.

In [ ]:
import sys
from importlib.util import find_spec

# Pyodide and xeus-python both report sys.platform "emscripten"; only Pyodide ships the pyodide package.
if sys.platform != "emscripten":
    KERNEL = "cpython"
else:
    try:
        has_pyodide = find_spec("pyodide") is not None
    except ValueError:  # xeus-python's pyodide stub has __spec__ set to None
        has_pyodide = False
    KERNEL = "pyodide" if has_pyodide else "xpython"

if KERNEL == "pyodide" and find_spec("a2ui_jupyter") is None:
    import piplite

    await piplite.install(["rpds-py", "a2ui-jupyter"])

from a2ui_jupyter import A2UISurface

BASIC_CATALOG_ID = "https://a2ui.org/specification/v1_0/catalogs/basic/catalog.json"


def new_surface(surface_id, *, send_data_model=False, **widget_kwargs):
    surface = A2UISurface(**widget_kwargs)
    create = {"surfaceId": surface_id, "catalogId": BASIC_CATALOG_ID}
    if send_data_model:
        create["sendDataModel"] = True
    surface.send({"version": "1.0", "createSurface": create})
    return surface

In [ ]:
surface = new_surface("text-and-layout")
surface.send(
    {
        "version": "1.0",
        "updateDataModel": {
            "surfaceId": "text-and-layout",
            "path": "/greeting",
            "value": "Hello from the data model",
        },
    }
)
surface.send(
    {
        "version": "1.0",
        "updateComponents": {
            "surfaceId": "text-and-layout",
            "components": [
                {"id": "root", "component": "Card", "child": "main"},
                {
                    "id": "main",
                    "component": "Column",
                    "children": ["heading", "intro", "divider", "row", "greeting"],
                },
                {"id": "heading", "component": "Text", "text": "## Layout basics"},
                {
                    "id": "intro",
                    "component": "Text",
                    "text": "Columns stack children vertically. Rows place them side by side.",
                    "variant": "body",
                },
                {"id": "divider", "component": "Divider"},
                {
                    "id": "row",
                    "component": "Row",
                    "children": ["left", "right"],
                    "justify": "spaceBetween",
                },
                {"id": "left", "component": "Text", "text": "Left"},
                {"id": "right", "component": "Text", "text": "Right", "variant": "caption"},
                {"id": "greeting", "component": "Text", "text": {"@path": "/greeting"}},
            ],
        },
    }
)
surface

The last text component is bound to `/greeting` in the data model. Run the next cell to change that value. The rendered text updates without rebuilding the surface.

In [ ]:
surface.send(
    {
        "version": "1.0",
        "updateDataModel": {
            "surfaceId": "text-and-layout",
            "path": "/greeting",
            "value": "Updated from a later cell",
        },
    }
)